In [1]:
!pip install torch torchvision -q
!pip install kornia -q
!pip install opencv-python -q
!pip install tqdm matplotlib -q
!pip install hdbscan -q
!pip install faiss-cpu -q
!pip install git+https://github.com/cvg/Hierarchical-Localization.git -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 17.2 MB/s eta 0:00:00


In [2]:
import os
import gc
import copy
import time
import traceback
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
import faiss
import hdbscan

from pathlib import Path
from tqdm import tqdm
from PIL import Image, ExifTags
from scipy.spatial.transform import Rotation
from collections import defaultdict

from hloc import extract_features, match_features, reconstruction
from hloc.utils.read_write_model import read_model

In [3]:
# Locate data
for _candidate in [
    Path("/kaggle/input/image-matching-challenge-2025"),
    Path("/kaggle/input/competitions/image-matching-challenge-2025"),
]:
    if _candidate.exists():
        DATA_DIR = _candidate
        break
else:
    raise FileNotFoundError("Cannot locate competition data directory.")

print(f"DATA_DIR = {DATA_DIR}")

OUTPUT_DIR   = Path("/kaggle/working")
FEATURES_DIR = OUTPUT_DIR / "features"
MATCHES_DIR  = OUTPUT_DIR / "matches"
SFM_DIR      = OUTPUT_DIR / "sfm"
PAIRS_DIR    = OUTPUT_DIR / "pairs"

for d in [FEATURES_DIR, MATCHES_DIR, SFM_DIR, PAIRS_DIR]:
    d.mkdir(exist_ok=True, parents=True)

# Hyperparameters
TOP_K              = 50       # FAISS neighbours per image (was 30)
RESIZE_MAX         = 1024     # local feature resolution (was 1024)
MAX_PAIRS_PER_DS   = 8000     # cap total pairs per dataset
EXHAUSTIVE_THRESH  = 80       # use exhaustive matching if cluster ≤ this size
TIME_BUDGET        = 8.0 * 3600  # 8 hours in seconds (leave 1h margin)

NAN_ROT = "nan;nan;nan;nan;nan;nan;nan;nan;nan"
NAN_T   = "nan;nan;nan"

START_TIME = time.time()
# Hyperparameters
TOP_K              = 50       # FAISS neighbours per image (was 30)
RESIZE_MAX         = 1600     # local feature resolution (was 1024)
MAX_PAIRS_PER_DS   = 8000     # cap total pairs per dataset
EXHAUSTIVE_THRESH  = 80       # use exhaustive matching if cluster ≤ this size
TIME_BUDGET        = 8.0 * 3600  # 8 hours in seconds (leave 1h margin)

NAN_ROT = "nan;nan;nan;nan;nan;nan;nan;nan;nan"
NAN_T   = "nan;nan;nan"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

START_TIME = time.time()

DATA_DIR = /kaggle/input/competitions/image-matching-challenge-2025
Device: cuda


In [4]:
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
datasets = sample_submission["dataset"].unique()
print(f"Datasets ({len(datasets)}): {list(datasets)}")

Datasets (13): ['ETs', 'amy_gardens', 'fbk_vineyard', 'imc2023_haiper', 'imc2023_heritage', 'imc2023_theather_imc2024_church', 'imc2024_dioscuri_baalshamin', 'imc2024_lizard_pond', 'pt_brandenburg_british_buckingham', 'pt_piazzasanmarco_grandplace', 'pt_sacrecoeur_trevi_tajmahal', 'pt_stpeters_stpauls', 'stairs']


In [5]:
print("Loading DINOv2 ViT-S/14 …")
dinov2 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
dinov2 = dinov2.to(device).eval()
print("DINOv2 ready.")

Loading DINOv2 ViT-S/14 …
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 380MB/s]


DINOv2 ready.


In [6]:
# A) Dataset / image discovery
def find_dataset_dir(data_dir: Path, dataset: str):
    candidates = [
        data_dir / dataset,
        data_dir / "test"  / dataset,
        data_dir / "train" / dataset,
    ]
    for root in [data_dir, data_dir / "test", data_dir / "train"]:
        if root.exists():
            for sub in root.iterdir():
                if sub.is_dir() and sub.name == dataset:
                    candidates.append(sub)
    for p in candidates:
        if p.exists() and p.is_dir():
            return p
    return None


def get_image_paths(directory: Path) -> list:
    exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
    return sorted(
        p for p in directory.rglob("*")
        if p.is_file() and p.suffix in exts and "LICENSE" not in p.name
    )


# B) Image pre-processing
def fix_rotation(img: Image.Image) -> Image.Image:
    try:
        exif = img._getexif()
        if exif is None:
            return img
        key = next(k for k, v in ExifTags.TAGS.items() if v == "Orientation")
        o = exif.get(key, 1)
        if   o == 3: img = img.rotate(180, expand=True)
        elif o == 6: img = img.rotate(-90, expand=True)
        elif o == 8: img = img.rotate( 90, expand=True)
    except Exception:
        pass
    return img


# C) Global feature extraction (DINOv2)
def extract_global_features(image_paths: list, dataset_dir: Path):
    feats, names = [], []
    for path in tqdm(image_paths, desc="  DINOv2"):
        try:
            img = Image.open(path).convert("RGB")
            img = fix_rotation(img)
            img = img.resize((224, 224))
            arr = np.array(img, dtype=np.float32) / 255.0
            arr = arr.transpose(2, 0, 1)  # HWC → CHW
            with torch.no_grad():
                feat = dinov2(torch.from_numpy(arr).unsqueeze(0).to(device))
            f = feat.cpu().numpy().flatten()
            f = f / (np.linalg.norm(f) + 1e-8)  # L2 normalize
            feats.append(f)
            names.append(str(path.relative_to(dataset_dir)))
        except Exception as e:
            print(f"Warning: skipping {path.name}: {e}")
    if feats:
        global_feats = np.array(feats, dtype=np.float32)
    else:
        global_feats = np.empty((0, 384), dtype=np.float32)
    return names, global_feats


# D) HDBSCAN clustering with noise reassignment
def cluster_images(global_feats: np.ndarray, n_images: int):
    if n_images < 20:
        min_cluster_sz, min_samples = 2, 1
    elif n_images < 60:
        min_cluster_sz, min_samples = 3, 2
    elif n_images < 150:
        min_cluster_sz, min_samples = 5, 3
    else:
        min_cluster_sz, min_samples = 8, 4

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_sz,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
    )
    labels = clusterer.fit_predict(global_feats)
    unique_clusters = sorted(set(labels) - {-1})

    # If no clusters found, put everything in one cluster
    if len(unique_clusters) == 0:
        print("HDBSCAN found 0 clusters -> assigning all to cluster 0")
        return np.zeros(n_images, dtype=int)

    n_noise_before = int((labels == -1).sum())

    # Reassign noise points to nearest cluster center
    if n_noise_before > 0:
        centroids = []
        for c in unique_clusters:
            mask = labels == c
            centroid = global_feats[mask].mean(axis=0)
            centroid = centroid / (np.linalg.norm(centroid) + 1e-8)
            centroids.append(centroid)
        centroids = np.array(centroids, dtype=np.float32)

        noise_idx = np.where(labels == -1)[0]
        noise_feats = global_feats[noise_idx]
        sims = noise_feats @ centroids.T

        for i, idx in enumerate(noise_idx):
            best_pos = np.argmax(sims[i])
            if sims[i, best_pos] > 0.3:
                labels[idx] = unique_clusters[best_pos]

    n_noise_after = int((labels == -1).sum())
    n_clusters = len(set(labels) - {-1})
    print(f"HDBSCAN -> {n_clusters} clusters, "
          f"{n_noise_before} noise -> {n_noise_after} after reassignment")
    return labels


# E) Pair generation (per-cluster, FAISS or exhaustive)
def generate_pairs(names, global_feats, labels):
    """Generate pairs: exhaustive for small clusters, FAISS top-k + cross-cluster bridges."""
    all_pairs = set()
    unique_clusters = sorted(set(labels) - {-1})

    for c in unique_clusters:
        cluster_idx = np.where(labels == c)[0]
        n_cluster = len(cluster_idx)
        if n_cluster < 2:
            continue

        cluster_names = [names[i] for i in cluster_idx]
        cluster_feats = global_feats[cluster_idx]

        if n_cluster <= EXHAUSTIVE_THRESH:
            for i in range(n_cluster):
                for j in range(i + 1, n_cluster):
                    all_pairs.add((min(cluster_names[i], cluster_names[j]),
                                   max(cluster_names[i], cluster_names[j])))
        else:
            k = min(TOP_K + 1, n_cluster)
            feats_f32 = cluster_feats.astype(np.float32).copy()
            faiss.normalize_L2(feats_f32)
            index = faiss.IndexFlatIP(feats_f32.shape[1])
            index.add(feats_f32)
            _, indices = index.search(feats_f32, k)
            for i in range(n_cluster):
                for j_pos in range(1, k):
                    j = int(indices[i, j_pos])
                    if 0 <= j < n_cluster and j != i:
                        a, b = cluster_names[i], cluster_names[j]
                        all_pairs.add((min(a, b), max(a, b)))

    # Cross-cluster bridges: top-10 global NN across clusters
    if len(names) > 1 and len(unique_clusters) > 1:
        feats_all = global_feats.astype(np.float32).copy()
        faiss.normalize_L2(feats_all)
        k_global = min(11, len(names))
        index_all = faiss.IndexFlatIP(feats_all.shape[1])
        index_all.add(feats_all)
        _, indices_all = index_all.search(feats_all, k_global)

        for i in range(len(names)):
            for j_pos in range(1, k_global):
                j = int(indices_all[i, j_pos])
                if 0 <= j < len(names) and labels[i] != labels[j]:
                    a, b = names[i], names[j]
                    all_pairs.add((min(a, b), max(a, b)))

    pairs = sorted(all_pairs)
    if len(pairs) > MAX_PAIRS_PER_DS:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(pairs), MAX_PAIRS_PER_DS, replace=False)
        pairs = [pairs[i] for i in sorted(idx)]

    return pairs


# F) Extract poses from COLMAP reconstruction
def extract_poses_from_reconstruction(sfm_dir: Path):
    """Read COLMAP model -> dict: image_name -> (rot_str, t_str)."""
    poses = {}
    try:
        cameras, images, points3D = None, None, None

        # Try reading directly from sfm_dir
        for fmt in [".bin", ".txt"]:
            cam_file = sfm_dir / f"cameras{fmt}"
            img_file = sfm_dir / f"images{fmt}"
            if cam_file.exists() and img_file.exists():
                cameras, images, points3D = read_model(str(sfm_dir), ext=fmt)
                break

        # Try numbered subdirectories
        if images is None:
            for sub in sorted(sfm_dir.iterdir()):
                if sub.is_dir():
                    for fmt in [".bin", ".txt"]:
                        cam_file = sub / f"cameras{fmt}"
                        img_file = sub / f"images{fmt}"
                        if cam_file.exists() and img_file.exists():
                            cameras, images, points3D = read_model(str(sub), ext=fmt)
                            break
                    if images is not None:
                        break

        if images is None:
            return poses

        for img_id, img_data in images.items():
            name = img_data.name
            qvec = img_data.qvec   # (qw, qx, qy, qz)
            tvec = img_data.tvec   # (tx, ty, tz)

            # scipy expects (qx, qy, qz, qw)
            R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
            R_mat = R.as_matrix()

            rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
            t_str   = ";".join(f"{v:.10f}" for v in tvec)
            poses[name] = (rot_str, t_str)

    except Exception as e:
        print(f"Warning: error reading model: {e}")
        traceback.print_exc()

    return poses

# G) Extract poses from the pycolmap model object (RELIABLE)
def extract_poses_from_model(model):
    """Extract poses directly from pycolmap.Reconstruction object."""
    poses = {}
    if model is None:
        return poses
    
    try:
        # Get all registered images
        try:
            images = model.images
        except AttributeError:
            return poses
        
        for img_id in images:
            try:
                img = images[img_id]
            except (KeyError, TypeError):
                continue
            
            name = img.name
            R_mat = None
            tvec = None
            
            # Method 1: pycolmap >= 0.6 (cam_from_world)
            try:
                cfw = img.cam_from_world
                R_mat = np.array(cfw.rotation.matrix())
                tvec = np.array(cfw.translation)
            except AttributeError:
                pass
            
            # Method 2: pycolmap 0.5 (rotmat + tvec)
            if R_mat is None:
                try:
                    R_mat = np.array(img.rotmat())
                    tvec = np.array(img.tvec)
                except AttributeError:
                    pass
            
            # Method 3: older pycolmap (qvec + tvec)
            if R_mat is None:
                try:
                    qvec = img.qvec
                    R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
                    R_mat = R.as_matrix()
                    tvec = np.array(img.tvec)
                except AttributeError:
                    pass
            
            # Method 4: projection_matrix fallback
            if R_mat is None:
                try:
                    P = np.array(img.projection_matrix())
                    R_mat = P[:3, :3]
                    tvec = P[:3, 3]
                except Exception:
                    pass
            
            if R_mat is None or tvec is None:
                continue
            
            rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
            t_str = ";".join(f"{v:.10f}" for v in tvec)
            poses[name] = (rot_str, t_str)
    
    except Exception as e:
        print(f"Warning: model extraction failed: {e}")
        traceback.print_exc()
    
    return poses

import pycolmap

def _read_single_model_dir(model_dir):
    """Read poses from one COLMAP model directory."""
    poses = {}
    model_dir = Path(model_dir)
    
    # Method 1: pycolmap.Reconstruction (handles all binary formats)
    try:
        rec = pycolmap.Reconstruction(str(model_dir))
        for img_id in rec.images:
            img = rec.images[img_id]
            name = img.name
            R_mat, tvec = None, None
            
            try:
                cfw = img.cam_from_world
                R_mat = np.array(cfw.rotation.matrix())
                tvec  = np.array(cfw.translation)
            except Exception:
                pass
            if R_mat is None:
                try:
                    R_mat = np.array(img.rotmat())
                    tvec  = np.array(img.tvec)
                except Exception:
                    pass
            if R_mat is None:
                try:
                    qvec = np.array(img.qvec)
                    R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
                    R_mat = R.as_matrix()
                    tvec  = np.array(img.tvec)
                except Exception:
                    pass
            
            if R_mat is not None and tvec is not None:
                rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
                t_str   = ";".join(f"{v:.10f}" for v in tvec)
                poses[name] = (rot_str, t_str)
        if poses:
            return poses
    except Exception:
        pass
    
    # Method 2: hloc read_model fallback
    try:
        for ext in [".bin", ".txt"]:
            if (model_dir / f"cameras{ext}").exists():
                cameras, images, _ = read_model(str(model_dir), ext=ext)
                for img_id, img_data in images.items():
                    qvec = img_data.qvec
                    tvec = img_data.tvec
                    R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
                    R_mat = R.as_matrix()
                    rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
                    t_str   = ";".join(f"{v:.10f}" for v in tvec)
                    poses[img_data.name] = (rot_str, t_str)
                if poses:
                    return poses
    except Exception:
        pass
    
    return poses


def read_poses_from_colmap(sfm_dir):
    """Read ALL sub-models from COLMAP reconstruction output."""
    sfm_dir = Path(sfm_dir)
    all_poses = {}
    
    # 1) Read numbered sub-models from models/ directory
    models_dir = sfm_dir / "models"
    if models_dir.exists():
        for sub in sorted(models_dir.iterdir()):
            if sub.is_dir():
                poses = _read_single_model_dir(sub)
                new = 0
                for name, pose in poses.items():
                    if name not in all_poses:
                        all_poses[name] = pose
                        new += 1
                if poses:
                    print(f"    models/{sub.name}: {len(poses)} registered, {new} new")
    
    # 2) Read from main sfm_dir (hloc copies "best" model here)
    main_poses = _read_single_model_dir(sfm_dir)
    new = 0
    for name, pose in main_poses.items():
        if name not in all_poses:
            all_poses[name] = pose
            new += 1
    if main_poses:
        print(f"    main dir: {len(main_poses)} registered, {new} new")
    
    return all_poses

def _read_poses_from_dir(model_dir: Path):
    """Read poses from a single COLMAP model directory."""
    poses = {}
    
    # Method 1: pycolmap.Reconstruction (most reliable)
    try:
        rec = pycolmap.Reconstruction(str(model_dir))
        try:
            reg_ids = rec.reg_image_ids()
        except AttributeError:
            reg_ids = list(rec.images.keys())
        
        for img_id in reg_ids:
            img = rec.images[img_id]
            name = img.name
            R_mat, tvec = None, None
            
            try:
                cfw = img.cam_from_world
                R_mat = np.array(cfw.rotation.matrix())
                tvec  = np.array(cfw.translation)
            except AttributeError:
                pass
            if R_mat is None:
                try:
                    R_mat = np.array(img.rotmat())
                    tvec  = np.array(img.tvec)
                except AttributeError:
                    pass
            if R_mat is None:
                try:
                    qvec = np.array(img.qvec)
                    R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
                    R_mat = R.as_matrix()
                    tvec  = np.array(img.tvec)
                except AttributeError:
                    pass
            
            if R_mat is not None and tvec is not None:
                rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
                t_str   = ";".join(f"{v:.10f}" for v in tvec)
                poses[name] = (rot_str, t_str)
        
        if poses:
            return poses
    except Exception:
        pass
    
    # Method 2: hloc read_model fallback
    try:
        for ext in [".bin", ".txt"]:
            if (model_dir / f"cameras{ext}").exists():
                cameras, images, _ = read_model(str(model_dir), ext=ext)
                if images:
                    for img_id, img_data in images.items():
                        qvec = img_data.qvec
                        tvec = img_data.tvec
                        R = Rotation.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
                        R_mat = R.as_matrix()
                        rot_str = ";".join(f"{v:.10f}" for v in R_mat.flatten())
                        t_str   = ";".join(f"{v:.10f}" for v in tvec)
                        poses[img_data.name] = (rot_str, t_str)
                    return poses
    except Exception:
        pass
    
    return poses


def extract_all_poses(sfm_dir: Path):
    """Read ALL sub-models from COLMAP reconstruction."""
    all_poses = {}
    
    # Priority 1: Read each sub-model from models/ directory
    models_dir = sfm_dir / "models"
    if models_dir.exists():
        sub_dirs = sorted(
            [d for d in models_dir.iterdir() if d.is_dir()],
            key=lambda d: int(d.name) if d.name.isdigit() else 0
        )
        for sub in sub_dirs:
            poses = _read_poses_from_dir(sub)
            new = 0
            for name, pose in poses.items():
                if name not in all_poses:
                    all_poses[name] = pose
                    new += 1
            if poses:
                print(f"models/{sub.name}: {len(poses)} registered, {new} new")
    
    # Priority 2: Also read from main sfm_dir (hloc saves "best" model here)
    main_poses = _read_poses_from_dir(sfm_dir)
    new = 0
    for name, pose in main_poses.items():
        if name not in all_poses:
            all_poses[name] = pose
            new += 1
    if main_poses:
        print(f"main dir: {len(main_poses)} registered, {new} new")
    
    return all_poses

In [7]:
import pycolmap
import shutil

all_rows = []

for ds_idx, dataset in enumerate(datasets):
    elapsed  = time.time() - START_TIME
    print(f"\n[{ds_idx+1}/{len(datasets)}] Dataset: {dataset}")
    print(f"Time elapsed: {elapsed/60:.1f}min")

    # 1. Locate dataset 
    dataset_dir = find_dataset_dir(DATA_DIR, dataset)
    if dataset_dir is None:
        print("Dataset directory not found")
        continue
    print(f"Path: {dataset_dir}")

    # 2. Collect images 
    image_paths = get_image_paths(dataset_dir)
    n_images = len(image_paths)
    if n_images < 2:
        print(f"Only {n_images} image(s) — skipping")
        continue
    print(f"Images: {n_images}")

    # 3. Output dirs
    ds_feat_dir  = FEATURES_DIR / dataset
    ds_match_dir = MATCHES_DIR  / dataset
    ds_sfm_dir   = SFM_DIR      / dataset
    pairs_file   = PAIRS_DIR    / f"{dataset}_pairs.txt"
    for d in [ds_feat_dir, ds_match_dir, ds_sfm_dir]:
        d.mkdir(exist_ok=True, parents=True)

    # 4. DINOv2 global features (Fast, always run to get scene_map)
    names, global_feats = extract_global_features(image_paths, dataset_dir)
    if not names:
        print("No global features extracted")
        continue

    # 5. HDBSCAN clustering 
    labels = cluster_images(global_feats, len(names))
    scene_map = {}
    unique_labels = sorted(set(labels))
    for i, name in enumerate(names):
        scene_map[name] = f"scene_{labels[i]}"
    
    # 6. Generate pairs 
    pairs = generate_pairs(names, global_feats, labels)
    if not pairs:
        print("No pairs generated")
        continue
    print(f"Pairs: {len(pairs)}")

    # Write pairs file
    with open(pairs_file, "w") as f:
        for a, b in pairs:
            f.write(f"{a} {b}\n")

    # 7. Extract features (SKIP IF EXISTS)
    feature_conf = copy.deepcopy(extract_features.confs["aliked-n16"])
    feature_conf["preprocessing"]["resize_max"] = RESIZE_MAX
    feature_conf['device'] = device
    
    feature_path = extract_features.main(
        feature_conf, dataset_dir, ds_feat_dir, overwrite=False # Changed to False
    )
    print(f"Features: {feature_path.name}")

    # 8. Match ALL pairs (SKIP IF EXISTS)
    matcher_conf = copy.deepcopy(match_features.confs["aliked+lightglue"])
    matcher_conf['device'] = device
    
    match_output = ds_match_dir / f"{matcher_conf['output']}.h5"
    match_path = match_features.main(
        matcher_conf, pairs_file,
        features=feature_path, matches=match_output, overwrite=False # Changed to False
    )
    print(f"Matches: {match_path.name}")

    # 9. Per-cluster SfM (SKIP IF EXISTS)
    all_poses = {}
    unique_clusters = sorted(set(labels) - {-1})

    for c in unique_clusters:
        cluster_idx = [i for i, l in enumerate(labels) if l == c]
        cluster_names = [names[i] for i in cluster_idx]
        if len(cluster_names) < 2:
            continue

        # Filter pairs to within this cluster only
        cluster_set = set(cluster_names)
        cluster_pairs = [(a, b) for a, b in pairs
                         if a in cluster_set and b in cluster_set]
        if not cluster_pairs:
            continue

        # Write cluster-specific pairs file
        cluster_pairs_file = PAIRS_DIR / f"{dataset}_c{c}.txt"
        with open(cluster_pairs_file, "w") as f:
            for a, b in cluster_pairs:
                f.write(f"{a} {b}\n")

        cluster_sfm = ds_sfm_dir / f"cluster_{c}"
        
        # --- RESUME LOGIC ---
        # Try to read existing poses first
        existing_poses = read_poses_from_colmap(cluster_sfm)
        
        # If we found poses, assume it's done and skip
        if existing_poses:
            print(f"  Cluster {c}: Found existing reconstruction ({len(existing_poses)} poses). Skipping.")
            all_poses.update(existing_poses)
            continue
        
        # If not, clean directory and run SfM
        if cluster_sfm.exists():
            shutil.rmtree(cluster_sfm)
        cluster_sfm.mkdir(parents=True)

        try:
            reconstruction.main(
                cluster_sfm, dataset_dir,
                cluster_pairs_file, feature_path, match_path,
                image_list=cluster_names,
            )
        except TypeError:
            try:
                reconstruction.main(
                    cluster_sfm, dataset_dir,
                    cluster_pairs_file, feature_path, match_path,
                )
            except Exception as e2:
                print(f"  Cluster {c}: SfM failed — {e2}")
                continue
        except Exception as e:
            print(f"  Cluster {c}: SfM failed — {e}")
            continue

        # Read new poses
        cluster_poses = read_poses_from_colmap(cluster_sfm)
        n_new = 0
        for name in cluster_names:
            if name in cluster_poses and name not in all_poses:
                all_poses[name] = cluster_poses[name]
                n_new += 1
        print(f"  Cluster {c}: {n_new}/{len(cluster_names)} registered")

    # 10. Finalize for this dataset
    print(f"Total poses: {len(all_poses)}/{n_images}")

    # 11. Write rows 
    sub_df = sample_submission[sample_submission["dataset"] == dataset]
    n_valid = 0
    for _, row in sub_df.iterrows():
        name = row["image"]
        if name in all_poses:
            rot_str, t_str = all_poses[name]
            n_valid += 1
        else:
            rot_str = NAN_ROT
            t_str   = NAN_T

        all_rows.append({
            "dataset":            dataset,
            "image":              name,
            "scene":              scene_map.get(name, "scene_0"),
            "rotation_matrix":    rot_str,
            "translation_vector": t_str,
        })

    print(f"{n_valid}/{len(sub_df)} registered ({100*n_valid/len(sub_df):.1f}%)")

    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nTotal time: {(time.time()-START_TIME)/60:.1f} min")


[1/13] Dataset: ETs
Time elapsed: 0.0min
Path: /kaggle/input/competitions/image-matching-challenge-2025/test/ETs
Images: 22


  DINOv2: 100%|██████████| 22/22 [00:02<00:00, 10.32it/s]
[2026/03/17 07:57:57 hloc INFO] Extracting local features with configuration:
{'device': 'cuda',
 'model': {'model_name': 'aliked-n16', 'name': 'aliked'},
 'output': 'feats-aliked-n16',
 'preprocessing': {'grayscale': False, 'resize_max': 1600}}
[2026/03/17 07:57:57 hloc INFO] Found 22 images in root /kaggle/input/competitions/image-matching-challenge-2025/test/ETs.


HDBSCAN -> 3 clusters, 1 noise -> 0 after reassignment
Pairs: 143
Downloading: "https://github.com/Shiaoming/ALIKED/raw/main/models/aliked-n16.pth" to /root/.cache/torch/hub/checkpoints/aliked-n16.pth


100%|██████████| 2.61M/2.61M [00:00<00:00, 30.7MB/s]
100%|██████████| 22/22 [00:01<00:00, 12.48it/s]
[2026/03/17 07:58:08 hloc INFO] Finished exporting features.
[2026/03/17 07:58:08 hloc INFO] Matching local features with configuration:
{'device': 'cuda',
 'model': {'features': 'aliked', 'name': 'lightglue'},
 'output': 'matches-aliked-lightglue'}


Features: feats-aliked-n16.h5
Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/aliked_lightglue.pth" to /root/.cache/torch/hub/checkpoints/aliked_lightglue_v0-1_arxiv.pth


...

    main dir: 33 registered, 33 new
  Cluster 0: 33/39 registered


[2026/03/17 13:00:20 hloc INFO] Importing features into the database...
100%|██████████| 6/6 [00:00<00:00, 973.31it/s]
[2026/03/17 13:00:20 hloc INFO] Importing matches into the database...
100%|██████████| 15/15 [00:00<00:00, 944.41it/s]
[2026/03/17 13:00:20 hloc INFO] Performing geometric verification of the matches...
[2026/03/17 13:00:21 hloc INFO] Running 3D reconstruction...
Reconstruction 0:  33%|███▎      | 2/6 [00:00<00:00, 77.35images/s, registered]
[2026/03/17 13:00:21 hloc INFO] Reconstructed 1 model(s).
[2026/03/17 13:00:21 hloc INFO] Largest model is #0 with 2 images.
[2026/03/17 13:00:21 hloc INFO] Reconstruction statistics:
Reconstruction:
	num_rigs = 2
	num_cameras = 2
	num_frames = 2
	num_reg_frames = 2
	num_images = 2
	num_points3D = 40
	num_observations = 80
	mean_track_length = 2
	mean_observations_per_image = 40
	mean_reprojection_error = 0.242217
	num_input_images = 6
E20260317 13:00:21.125194 134774516892800 reconstruction.cc:974] rigs, cameras, frames, images, 

    main dir: 2 registered, 2 new
  Cluster 1: 2/6 registered


[2026/03/17 13:00:21 hloc INFO] Importing features into the database...
100%|██████████| 6/6 [00:00<00:00, 972.78it/s]
[2026/03/17 13:00:21 hloc INFO] Importing matches into the database...
100%|██████████| 15/15 [00:00<00:00, 947.82it/s]
[2026/03/17 13:00:21 hloc INFO] Performing geometric verification of the matches...
[2026/03/17 13:00:21 hloc INFO] Running 3D reconstruction...
E20260317 13:00:21.480313 134774516892800 sfm.cc:279] Failed to create any sparse model
[2026/03/17 13:00:21 hloc ERROR] Could not reconstruct any model!
E20260317 13:00:21.481320 134774516892800 reconstruction.cc:974] rigs, cameras, frames, images, points3D files do not exist at "/kaggle/working/sfm/stairs/cluster_2"


  Cluster 2: 0/6 registered
Total poses: 35/51
35/51 registered (68.6%)

Total time: 302.5 min


In [8]:
# Create lookup: (dataset, image_name) → (rotation, translation, scene)
pose_lookup = {}
for row in all_rows:
    key = (row["dataset"], row["image"])
    pose_lookup[key] = (
        row["rotation_matrix"],
        row["translation_vector"],
        row["scene"],
    )

# Start from the ORIGINAL sample_submission to keep correct image_id values
submission = sample_submission.copy()

n_filled = 0
for idx, row in submission.iterrows():
    key = (row["dataset"], row["image"])
    if key in pose_lookup:
        rot, t, scene = pose_lookup[key]
        submission.at[idx, "rotation_matrix"]    = rot
        submission.at[idx, "translation_vector"] = t
        submission.at[idx, "scene"]              = scene
        if "nan" not in rot:
            n_filled += 1
    else:
        submission.at[idx, "rotation_matrix"]    = NAN_ROT
        submission.at[idx, "translation_vector"] = NAN_T
        submission.at[idx, "scene"]              = "scene_0"

print(f"Submission: {len(submission)} rows, {n_filled} with valid poses "
      f"({100*n_filled/len(submission):.1f}%)")

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head(30))

Submission: 1945 rows, 1713 with valid poses (88.1%)
Saved submission.csv
                                   image_id      dataset    scene  \
0   ETs_another_et_another_et001.png_public          ETs  scene_0   
1   ETs_another_et_another_et002.png_public          ETs  scene_0   
2   ETs_another_et_another_et003.png_public          ETs  scene_0   
3   ETs_another_et_another_et004.png_public          ETs  scene_0   
4   ETs_another_et_another_et005.png_public          ETs  scene_0   
5   ETs_another_et_another_et006.png_public          ETs  scene_0   
6   ETs_another_et_another_et007.png_public          ETs  scene_0   
7   ETs_another_et_another_et008.png_public          ETs  scene_0   
8   ETs_another_et_another_et009.png_public          ETs  scene_0   
9   ETs_another_et_another_et010.png_public          ETs  scene_0   
10                  ETs_et_et000.png_public          ETs  scene_2   
11                  ETs_et_et001.png_public          ETs  scene_2   
12                  ETs_et_et

In [9]:
print(f"{'PERFORMANCE DIAGNOSTICS':^80}")

# Per-dataset breakdown
header = f"  {'Dataset':<48} {'Valid':>5} {'Total':>5} {'Rate':>7} {'Scenes':>6}  Status"
print(header)

ds_stats = []
for ds in datasets:
    ds_df   = submission[submission["dataset"] == ds]
    total   = len(ds_df)
    valid   = ds_df["rotation_matrix"].apply(lambda x: "nan" not in str(x)).sum()
    scenes  = ds_df["scene"].nunique()
    rate    = 100 * valid / total if total > 0 else 0

    ds_stats.append(dict(dataset=ds, total=total, valid=valid,
                         rate=rate, scenes=scenes))
    print(f"]{ds:<48} {valid:>5} {total:>5} {rate:>6.1f}% {scenes:>6}")

total_all = sum(s["total"] for s in ds_stats)
valid_all = sum(s["valid"] for s in ds_stats)
rate_all  = 100 * valid_all / total_all if total_all > 0 else 0
print(f"{'OVERALL':<48} {valid_all:>5} {total_all:>5} {rate_all:>6.1f}%")

# Scene distribution 
print(f"{'SCENE DISTRIBUTION':^80}")

for ds in datasets:
    ds_df = submission[submission["dataset"] == ds]
    scene_counts = ds_df.groupby("scene").agg(
        total=("image", "count"),
        valid=("rotation_matrix",
               lambda x: (x.apply(lambda v: "nan" not in str(v))).sum()),
    ).reset_index()
    print(f"{ds}:")
    for _, sc in scene_counts.iterrows():
        bar_len = int(20 * sc["valid"] / sc["total"]) if sc["total"] > 0 else 0
        print(f"{sc['scene']:<16} {sc['valid']:>3}/{sc['total']:<3}")
    print()

# Datasets needing improvement
worst = [s for s in ds_stats if s["rate"] < 80]
if worst:
    print(f"{'DATASETS NEEDING IMPROVEMENT':^80}")
    for s in sorted(worst, key=lambda x: x["rate"]):
        missing = s["total"] - s["valid"]
        print(f"{s['dataset']}: {s['rate']:.1f}% — "
              f"{missing} images unregistered")

# Sanity checks 
print(f"{'SANITY CHECKS':^80}")

sample_id = submission.iloc[0]["image_id"]
print(f"image_id format looks correct: '{sample_id}'")

dupes = submission.duplicated(subset=["image_id"]).sum()
print(f"Duplicate image_ids: {dupes}")

has_valid = submission["rotation_matrix"].apply(
    lambda x: "nan" not in str(x))
if has_valid.any():
    sample_rot = submission.loc[has_valid.idxmax(), "rotation_matrix"]
    sample_t   = submission.loc[has_valid.idxmax(), "translation_vector"]
    n_r = len(str(sample_rot).split(";"))
    n_t = len(str(sample_t).split(";"))
    print(f"Rotation values: {n_r} (expected 9)")
    print(f"Translation values: {n_t} (expected 3)")
    print(f"Sample rotation:{str(sample_rot)[:70]}...")
    print(f"Sample translation: {sample_t}")
else:
    print(f"NO valid poses in entire submission — pipeline is broken!")

all_scene0 = (submission["scene"] == "scene_0").all()
print(f"  {'All scene_0!' if all_scene0 else 'Yay'} "
      f"Scene diversity: {submission['scene'].nunique()} unique scenes")

print(f"\nTotal pipeline time: {(time.time() - START_TIME)/60:.1f} min")

                            PERFORMANCE DIAGNOSTICS                             
  Dataset                                          Valid Total    Rate Scenes  Status
]ETs                                                 19    22   86.4%      3
]amy_gardens                                        174   200   87.0%      4
]fbk_vineyard                                       160   163   98.2%      2
]imc2023_haiper                                      54    54  100.0%      2
]imc2023_heritage                                   111   209   53.1%      6
]imc2023_theather_imc2024_church                     75    76   98.7%      3
]imc2024_dioscuri_baalshamin                        126   138   91.3%      5
]imc2024_lizard_pond                                143   214   66.8%      3
]pt_brandenburg_british_buckingham                  224   225   99.6%      3
]pt_piazzasanmarco_grandplace                       167   168   99.4%      2
]pt_sacrecoeur_trevi_tajmahal                       225   225  

In [10]:
submission.head(30)

,image_id,dataset,scene,image,rotation_matrix,translation_vector
0,ETs_another_et_another_et001.png_public,ETs,scene_0,another_et_another_et001.png,0.9970962076;0.0678989057;0.0344803036;-0.0699...,-0.7313912904;-0.1775876108;0.6594401430
1,ETs_another_et_another_et002.png_public,ETs,scene_0,another_et_another_et002.png,0.9976483278;0.0681876918;0.0069464157;-0.0678...,-0.7489506347;-0.2432090239;0.6166598056
2,ETs_another_et_another_et003.png_public,ETs,scene_0,another_et_another_et003.png,0.9994538629;-0.0267464966;0.0194062081;0.0304...,-0.2260209596;3.6299436824;-3.4310965286
3,ETs_another_et_another_et004.png_public,ETs,scene_0,another_et_another_et004.png,0.9999708595;-0.0011515070;0.0075467959;0.0017...,0.1057754618;-3.0851465589;3.9332788182
4,ETs_another_et_another_et005.png_public,ETs,scene_0,another_et_another_et005.png,0.9874515855;0.0789014622;0.1367988504;-0.0936...,-0.6637802585;-0.1317974803;0.7364999138
5,ETs_another_et_another_et006.png_public,ETs,scene_0,another_et_another_et006.png,-0.4669583614;-0.4618323478;-0.7540959960;-0.4...,0.0003612955;0.0000707984;8.9993107923
6,ETs_another_et_another_et007.png_public,ETs,scene_0,another_et_another_et007.png,0.7425208387;0.1667185320;-0.6487431967;-0.317...,-0.9941547994;-0.0942053249;-0.0478702911
7,ETs_another_et_another_et008.png_public,ETs,scene_0,another_et_another_et008.png,0.5021870548;0.1484262693;-0.8519259384;-0.381...,-0.9382146801;-0.0066968556;-0.3449436175
8,ETs_another_et_another_et009.png_public,ETs,scene_0,another_et_another_et009.png,0.2120641559;0.1265769665;-0.9690237692;-0.459...,-0.7912363337;0.1093378937;-0.6008777456
9,ETs_another_et_another_et010.png_public,ETs,scene_0,another_et_another_et010.png,-0.3186607840;0.1077011842;-0.9417301947;-0.50...,-0.3782762026;0.3187277692;-0.8684240687


In [11]:
submission[30:100]

,image_id,dataset,scene,image,rotation_matrix,translation_vector
30,amy_gardens_peach_0008.png_public,amy_gardens,scene_1,peach_0008.png,0.9701059122;0.0385551792;-0.2395997023;-0.117...,-0.9372446143;-0.5116653183;0.5453998415
31,amy_gardens_peach_0009.png_public,amy_gardens,scene_1,peach_0009.png,-0.2258259837;0.0829477588;-0.9706298442;-0.24...,1.9105431626;-0.9420652747;5.1932705494
32,amy_gardens_peach_0010.png_public,amy_gardens,scene_3,peach_0010.png,nan;nan;nan;nan;nan;nan;nan;nan;nan,nan;nan;nan
33,amy_gardens_peach_0011.png_public,amy_gardens,scene_2,peach_0011.png,0.9552257786;0.0681380270;0.2879252006;-0.0421...,-2.7392534178;0.6441838421;-0.5005456180
34,amy_gardens_peach_0012.png_public,amy_gardens,scene_2,peach_0012.png,nan;nan;nan;nan;nan;nan;nan;nan;nan,nan;nan;nan
...,...,...,...,...,...,...
95,amy_gardens_peach_0073.png_public,amy_gardens,scene_0,peach_0073.png,0.9973584959;0.0626223336;-0.0368031802;-0.066...,-2.4076126415;0.3851929694;-0.3386450234
96,amy_gardens_peach_0074.png_public,amy_gardens,scene_2,peach_0074.png,0.9970792277;0.0668349327;0.0369608649;-0.0637...,4.4597403512;0.8784737035;-0.9465443467
97,amy_gardens_peach_0075.png_public,amy_gardens,scene_0,peach_0075.png,0.9999872362;-0.0001996945;-0.0050485289;0.000...,-2.1812286695;0.0963937949;0.8940872476
98,amy_gardens_peach_0076.png_public,amy_gardens,scene_1,peach_0076.png,0.9412016593;0.1395276324;-0.3076873027;-0.125...,-3.1583914013;0.0449137011;4.7796857783
